# 115 — Planificación y descomposición de tareas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Plan:** secuencia o grafo de sub-tareas propuesto ANTES de actuar. **Descomposición**
correcta = cada sub-tarea es verificable por sí misma, las dependencias son explícitas y
la composición implica el objetivo. **Hito:** predicado observable sobre el entorno que
confirma progreso real. **Parada:** predicado de éxito global + presupuesto máximo.

Dos estrategias: **plan-then-execute** (plan completo revisable antes de tocar el
entorno; rígido ante imprevistos) y **planificación entrelazada** (revisar tras cada
observación, estilo ReAct; riesgo de perder el rumbo global). La práctica combina
ambas: plan inicial + replanteo SOLO cuando una observación contradice un supuesto
(replan-on-failure), y solo en la rama afectada.

### 🛑 Tres finales legítimos

- **Éxito:** el predicado global se verificó contra observaciones.
- **Agotamiento:** presupuesto consumido → reportar estado parcial y qué falta.
- **Bloqueo:** dependencia externa (p. ej. aprobación humana) impide avanzar → escalar.

El laboratorio `workflow` ejecuta la versión mínima de un plan con hito bloqueante:
transiciones `received → validated → waiting_approval → completed`, donde `completed`
es inalcanzable sin la aprobación registrada.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Hay 3 transiciones: `received → validated → waiting_approval →
completed`. Estado inicial: `received`. El hito bloqueante es `waiting_approval`: el
evento hacia `completed` solo existe porque `approved: true` quedó registrado antes —
la evidencia del laboratorio lo dice explícitamente.

**Ejercicio 2 (una descomposición válida).** 1) Inventariar documentos Word (hito: lista
con N archivos y tamaños; 1 paso). 2) Convertir cada documento (dep: 1; hito: existe un
.md por cada .docx del inventario; 4 pasos). 3) Verificar enlaces e imágenes (dep: 2;
hito: 0 enlaces rotos según el verificador; 3 pasos). 4) Revisión humana del contenido
(dep: 3; hito: visto bueno registrado; BLOQUEANTE — la pérdida de contenido en la
conversión no la detecta ningún predicado automático). 5) Merge al repositorio (dep: 4;
hito: rama integrada y CI en verde; 2 pasos).

**Ejercicio 3.** Defectos: (a) ningún paso tiene hito verificable ("entender",
"mejorarlo", "bonito" no son predicados observables); (b) no hay dependencias explícitas
ni criterio de qué implica "todo funciona" (¿qué tests?); (c) granularidad no
presupuestable: "mejorarlo" puede ser infinito — no se puede estimar ni contener su
fallo. Reescritura mínima: 1) listar funciones sin tests (hito: lista concreta);
2) añadir tests que fijan el comportamiento actual (hito: N tests nuevos en verde);
3) refactorizar la función X (hito: tests siguen en verde, complejidad ciclomática
baja de A a B); 4) repetir por función con presupuesto por iteración.

**Ejercicio 4.** Si falla el hito 3 (enlaces rotos), se replantea solo la rama 3: se
inserta 3b "corregir rutas de imágenes en los .md afectados". Las sub-tareas 1 y 2
conservan su progreso (el inventario y la conversión ya están verificados); 4 y 5
simplemente esperan. El plan global sobrevive porque el fallo quedó contenido en la
única rama cuyo supuesto ("la conversión preserva rutas") resultó falso.

In [ ]:
result = run_lab("workflow", seed=115)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — máquina de estados reconstruida y verificada
result = run_lab("workflow", seed=115)
assert result["kind"] == "workflow"
events = result["result"]["events"]
cadena = [events[0]["from"]] + [e["to"] for e in events]
print(" -> ".join(cadena))
assert cadena[0] == "received" and cadena[-1] == "completed"
assert result["result"]["approved"] is True  # el hito bloqueante quedo registrado
assert {"from": "waiting_approval", "to": "completed"} in events


In [ ]:
# Ejercicio 2 — plan con hitos verificables (referencia)
plan = [
    {"id": 1, "subtarea": "inventariar .docx", "depende_de": [],
     "hito": "lista con N archivos", "presupuesto_pasos": 1},
    {"id": 2, "subtarea": "convertir a .md", "depende_de": [1],
     "hito": "un .md por cada .docx del inventario", "presupuesto_pasos": 4},
    {"id": 3, "subtarea": "verificar enlaces/imagenes", "depende_de": [2],
     "hito": "0 enlaces rotos", "presupuesto_pasos": 3},
    {"id": 4, "subtarea": "revision humana", "depende_de": [3],
     "hito": "visto bueno registrado", "presupuesto_pasos": None},  # BLOQUEANTE
    {"id": 5, "subtarea": "merge al repo", "depende_de": [4],
     "hito": "CI en verde", "presupuesto_pasos": 2},
]
# comprobacion simple: dependencias aciclicas y hacia atras
for t in plan:
    assert all(d < t["id"] for d in t["depende_de"])
print("plan valido:", len(plan), "sub-tareas; bloqueante -> id 4")


## Reflexión

1. En el laboratorio, `waiting_approval` está ANTES de `completed` por diseño. ¿Qué
   propiedad de la descomposición (hito bloqueante) representa y qué pasaría si el
   replanteo pudiera reordenar esa transición?
2. ¿Por qué "replantear en cada paso" destruye la ventaja del plan y cuál es el
   disparador correcto de un replanteo?
3. Da un ejemplo real donde la parada por agotamiento de presupuesto sea el resultado
   CORRECTO. ¿Qué debe contener el reporte de estado parcial para que otro agente (u
   humano) retome el trabajo sin repetirlo?